<a href="https://colab.research.google.com/github/MuhamedZepcanin/MuhamedZepcanin.github.io/blob/main/H5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#include "esp_camera.h"
#include <WiFi.h>
#include <WebServer.h>
#include "esp_wifi.h"
#include "esp_bt.h"

#define PART_BOUNDARY "frame"

// ================= WiFi =================
const char* ssid = "ESP32CAM";
const char* password = "12345678";

WebServer server(80);

// ================= Stats =================
volatile uint32_t frameCount = 0;
volatile uint32_t lastFpsTime = 0;
volatile float fps = 0;
volatile uint32_t lastFrameTime = 0;
volatile uint32_t latency = 0;

// ================= Camera Pins =================
#define PWDN_GPIO_NUM   32
#define RESET_GPIO_NUM  -1
#define XCLK_GPIO_NUM    0
#define SIOD_GPIO_NUM   26
#define SIOC_GPIO_NUM   27
#define Y9_GPIO_NUM     35
#define Y8_GPIO_NUM     34
#define Y7_GPIO_NUM     39
#define Y6_GPIO_NUM     36
#define Y5_GPIO_NUM     21
#define Y4_GPIO_NUM     19
#define Y3_GPIO_NUM     18
#define Y2_GPIO_NUM      5
#define VSYNC_GPIO_NUM  25
#define HREF_GPIO_NUM   23
#define PCLK_GPIO_NUM   22

// ================= HTML =================
const char HTML_PAGE[] PROGMEM = R"(
<!DOCTYPE html>
<html>
<head>
<title>ESP32 TurboCam</title>
<style>
body { margin:0; background:black; text-align:center; }
img { width:100%; max-width:480px; border:2px solid cyan; }
#stats {
  position:fixed; top:10px; left:10px;
  color:#00ffcc; font-family:monospace;
  background:rgba(0,0,0,0.5); padding:10px;
}
</style>
</head>
<body>
<div id="stats">FPS: -- | Latency: -- ms</div>
<img src="/stream">
<script>
setInterval(async ()=>{
  let r = await fetch('/stats');
  let j = await r.json();
  document.getElementById('stats').innerText =
    "FPS: " + j.fps + " | Latency: " + j.latency + " ms";
}, 500);
</script>
</body>
</html>
)";

// ================= Root =================
void handleRoot() {
    server.send_P(200, "text/html", HTML_PAGE);
}

// ================= Stats API =================
void handleStats() {
    String json = "{";
    json += "\"fps\":" + String(fps, 1) + ",";
    json += "\"latency\":" + String(latency);
    json += "}";
    server.send(200, "application/json", json);
}

// ================= Stream =================
void handleStream() {
    WiFiClient client = server.client();
    client.setNoDelay(true);

    client.print("HTTP/1.1 200 OK\r\n");
    client.print("Content-Type: multipart/x-mixed-replace;boundary=" PART_BOUNDARY "\r\n");
    client.print("Cache-Control: no-cache\r\n\r\n");

    char buf[64];

    while (client.connected()) {
        uint32_t start = millis();

        camera_fb_t* fb = esp_camera_fb_get();
        if (!fb) continue;

        client.print("--" PART_BOUNDARY "\r\n");
        client.print("Content-Type: image/jpeg\r\n");
        sprintf(buf, "Content-Length: %u\r\n\r\n", fb->len);
        client.print(buf);

        size_t sent = 0;
        const size_t CHUNK = 8192;

        while (sent < fb->len) {
            if (!client.connected()) break;
            size_t toSend = min(CHUNK, fb->len - sent);
            client.write(fb->buf + sent, toSend);
            sent += toSend;
        }

        client.print("\r\n");
        esp_camera_fb_return(fb);

        // ===== Stats =====
        uint32_t end = millis();
        latency = end - start;

        frameCount++;
        if (end - lastFpsTime >= 1000) {
            fps = frameCount;
            frameCount = 0;
            lastFpsTime = end;
        }

        yield();
    }
}

// ================= Camera Setup =================
void setupCamera() {
    camera_config_t cfg;

    cfg.ledc_channel = LEDC_CHANNEL_0;
    cfg.ledc_timer   = LEDC_TIMER_0;

    cfg.pin_d0 = Y2_GPIO_NUM;
    cfg.pin_d1 = Y3_GPIO_NUM;
    cfg.pin_d2 = Y4_GPIO_NUM;
    cfg.pin_d3 = Y5_GPIO_NUM;
    cfg.pin_d4 = Y6_GPIO_NUM;
    cfg.pin_d5 = Y7_GPIO_NUM;
    cfg.pin_d6 = Y8_GPIO_NUM;
    cfg.pin_d7 = Y9_GPIO_NUM;
    cfg.pin_xclk = XCLK_GPIO_NUM;
    cfg.pin_pclk = PCLK_GPIO_NUM;
    cfg.pin_vsync = VSYNC_GPIO_NUM;
    cfg.pin_href = HREF_GPIO_NUM;
    cfg.pin_sccb_sda = SIOD_GPIO_NUM;
    cfg.pin_sccb_scl = SIOC_GPIO_NUM;
    cfg.pin_pwdn = PWDN_GPIO_NUM;
    cfg.pin_reset = RESET_GPIO_NUM;

    // 🔥 TURBO SETTINGS
    cfg.xclk_freq_hz = 20000000;
    cfg.pixel_format = PIXFORMAT_JPEG;
    cfg.frame_size = FRAMESIZE_QQVGA;   // FAST
    cfg.jpeg_quality = 20;              // FAST ENCODE
    cfg.fb_count = 4;
    cfg.grab_mode = CAMERA_GRAB_LATEST;
    cfg.fb_location = CAMERA_FB_IN_PSRAM;

    if (esp_camera_init(&cfg) != ESP_OK) {
        Serial.println("Camera fail");
        return;
    }

    sensor_t* s = esp_camera_sensor_get();

    // 🔥 Disable slow auto features
    s->set_aec2(s, 0);
    s->set_exposure_ctrl(s, 0);
    s->set_aec_value(s, 300);
    s->set_gainceiling(s, GAINCEILING_2X);

    s->set_vflip(s, 1);
    s->set_hmirror(s, 1);
}

// ================= Setup =================
void setup() {
    Serial.begin(115200);

    WRITE_PERI_REG(RTC_CNTL_BROWN_OUT_REG, 0);

    esp_bt_controller_deinit();
    setCpuFrequencyMhz(240);

    setupCamera();

    WiFi.mode(WIFI_AP);
    WiFi.setSleep(false);
    esp_wifi_set_ps(WIFI_PS_NONE);

    WiFi.softAP(ssid, password, 1, 0, 4);

    Serial.println(WiFi.softAPIP());

    server.on("/", handleRoot);
    server.on("/stream", handleStream);
    server.on("/stats", handleStats);

    server.begin();
}

// ================= Loop =================
void loop() {
    server.handleClient();
}